# Scraping AI News Headlines

*Course:* MGS 3001 WHS01 — Python Programming for Business  
*Session:* Week 10-2, HTML Scraping & Unstructured Data  
*Date:* 30 April 2026

---

### What This Notebook Does

Scrapes article data (titles, dates, summaries, URLs) from **artificialintelligence-news.com** — a site with **no API**.

### Why scraping (not API)?

This site has no API. The only way to collect this data programmatically is to fetch the webpage and parse the HTML.

### Pipeline

```
Fetch page → Parse HTML (BeautifulSoup) → Find article elements → Extract fields → DataFrame → CSV
```

---
## Cell 1: Import Libraries

- `requests` — fetch the webpage (same function as API!)
- `BeautifulSoup` — parse HTML and search for elements
- `pandas` — organize data into a table
- `time` — delays between requests

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# If not installed:
# !pip install beautifulsoup4

print("Libraries imported!")

Libraries imported!


---
## Cell 2: Fetch the Page — Inspect Raw HTML

Same `requests.get()` as API sessions — but the server returns HTML, not JSON.

**Before running this cell**, open the site in your browser and use Developer Tools (F12 / right-click → Inspect) to look at the HTML structure. Find:
- Which tag contains each article? (`<article>`, `<div class="post">`, etc.)
- Where is the title? (usually `<h2>` or `<h3>` inside the article)
- Where is the date? (a `<time>` tag or a `<span>` with a date class)
- Where is the summary? (a `<p>` tag)
- Where is the link? (an `<a>` tag with `href`)

In [ ]:
# Fetch the AI News page
url = "https://www.artificialintelligence-news.com/"

# Add a User-Agent header so the site doesn't block us
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(url, headers=headers)
print(f"Status: {response.status_code}")
print(f"Content length: {len(response.text)} characters")

if response.status_code != 200:
    print(f"\u26a0\ufe0f Error fetching page. Try opening {url} in your browser to check if the site is accessible.")

---
## Cell 3: Parse with BeautifulSoup — Explore the Structure

Let’s parse the HTML and explore what elements are on the page.

**This is the investigation step** — we need to figure out the HTML structure before we can extract data.

**⚠️ Important:** The CSS selectors below are based on the site’s structure at the time this notebook was written. If the site has changed, you may need to update them using Developer Tools.

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# Step 1: Find all article elements
# Try common patterns — adjust based on what you see in Developer Tools
articles = soup.find_all("article")
print(f"Found {len(articles)} <article> elements")

# If no <article> tags, try other common patterns:
if len(articles) == 0:
    articles = soup.select(".post")
    print(f"Trying .post class: found {len(articles)} elements")

if len(articles) == 0:
    articles = soup.select(".entry")
    print(f"Trying .entry class: found {len(articles)} elements")

if len(articles) == 0:
    print("\n\u26a0\ufe0f Could not find article elements automatically.")
    print("Open the site in your browser, right-click an article title, select Inspect,")
    print("and look for the parent container element.")
    print("\nAll classes found on the page:")
    all_classes = set()
    for tag in soup.find_all(True):
        if tag.get('class'):
            all_classes.update(tag['class'])
    for c in sorted(all_classes)[:30]:
        print(f"  .{c}")

---
## Cell 4: Extract One Article — Test the Selectors

Before looping through all articles, let’s extract data from **one article** to make sure our selectors work.

**⚠️ Adjust the selectors below** based on what you found in Cell 3 and Developer Tools.

In [ ]:
if len(articles) > 0:
    # Take the first article
    article = articles[0]
    print("=== Raw HTML of first article (first 800 chars) ===")
    print(str(article)[:800])
    print("\n...")
    
    print("\n=== Attempting to extract fields ===")
    
    # Title: usually in <h2> or <h3>
    title_el = article.find("h2") or article.find("h3")
    title = title_el.text.strip() if title_el else "NO TITLE FOUND"
    print(f"Title: {title}")
    
    # Link: usually in the first <a> tag
    link_el = article.find("a", href=True)
    link = link_el.get("href", "") if link_el else "NO LINK FOUND"
    print(f"Link: {link}")
    
    # Date: look for <time> tag or element with date-related class
    date_el = article.find("time") or article.find(class_=lambda c: c and 'date' in c.lower() if c else False)
    date = date_el.text.strip() if date_el else "NO DATE FOUND"
    print(f"Date: {date}")
    
    # Summary: look for <p> tag (usually the first or second one)
    summary_el = article.find("p")
    summary = summary_el.text.strip() if summary_el else "NO SUMMARY FOUND"
    print(f"Summary: {summary[:150]}..." if len(summary) > 150 else f"Summary: {summary}")
else:
    print("No articles found. Please check Cell 3 and adjust selectors.")

---
## Cell 5: Define the Extraction Function

Once the selectors work for one article, wrap them in a reusable function.

Same pattern as the API notebook’s `extract_repo_info()` — a function that handles one item and returns a dictionary.

In [ ]:
def extract_article(article):
    """
    Extract data from a single article HTML element.
    Returns a dictionary with title, link, date, summary.
    
    Adjust selectors if the site structure changes.
    """
    # Title
    title_el = article.find("h2") or article.find("h3")
    title = title_el.text.strip() if title_el else ""
    
    # Link
    link_el = article.find("a", href=True)
    link = link_el.get("href", "") if link_el else ""
    
    # Date
    date_el = article.find("time")
    if not date_el:
        date_el = article.find(class_=lambda c: c and 'date' in c.lower() if c else False)
    date = date_el.text.strip() if date_el else ""
    
    # Summary
    summary_el = article.find("p")
    summary = summary_el.text.strip() if summary_el else ""
    
    return {
        "title": title,
        "date": date,
        "summary": summary,
        "url": link,
    }

# Test on first article
if len(articles) > 0:
    test = extract_article(articles[0])
    print("Test extraction:")
    for k, v in test.items():
        display = str(v)[:80] + "..." if len(str(v)) > 80 else str(v)
        print(f"  {k:10s} {display}")

---
## Cell 6: Extract All Articles on the Page

In [ ]:
# Extract data from all articles
all_articles = []

for i, article in enumerate(articles):
    data = extract_article(article)
    if data["title"]:  # Only keep articles with titles
        all_articles.append(data)

print(f"Extracted {len(all_articles)} articles from the page")

# Preview
for i, a in enumerate(all_articles[:5]):
    print(f"\n--- Article {i+1} ---")
    print(f"  Title: {a['title'][:70]}")
    print(f"  Date:  {a['date']}")
    print(f"  URL:   {a['url'][:70]}")

---
## Cell 7: Handle Pagination (Multiple Pages)

Most news sites have multiple pages. Let’s scrape a few pages to get more articles.

**⚠️ Adjust the URL pattern** based on how the site handles pagination (e.g., `/page/2/`, `?page=2`, etc.).

Check the site’s pagination links in your browser first.

In [ ]:
# Scrape multiple pages
max_pages = 3  # Adjust as needed
all_articles = []

for page_num in range(1, max_pages + 1):
    # Adjust URL pattern based on the site's pagination
    if page_num == 1:
        page_url = url
    else:
        page_url = f"{url}page/{page_num}/"  # Common WordPress pattern
    
    print(f"\nPage {page_num}: {page_url}")
    
    response = requests.get(page_url, headers=headers)
    
    if response.status_code != 200:
        print(f"  \u26a0\ufe0f Error: status {response.status_code}. Stopping.")
        break
    
    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.find_all("article")
    
    if len(articles) == 0:
        print("  No more articles. Stopping.")
        break
    
    for article in articles:
        data = extract_article(article)
        if data["title"]:
            all_articles.append(data)
    
    print(f"  Found {len(articles)} articles (total: {len(all_articles)})")
    
    # Be polite!
    time.sleep(2)

print(f"\n{'='*50}")
print(f"Total articles collected: {len(all_articles)}")

---
## Cell 8: Create DataFrame and Export

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_articles)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print()

# Preview
df.head(10)

In [ ]:
# Quick stats
print("=== Dataset Summary ===")
print(f"Total articles: {len(df)}")
print(f"Articles with dates: {df['date'].astype(bool).sum()}")
print(f"Articles with summaries: {df['summary'].astype(bool).sum()}")
print(f"Average summary length: {df['summary'].str.len().mean():.0f} characters")

In [ ]:
# Export to CSV
output_file = "ai_news_articles.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\u2705 Saved to: {output_file}")
print(f"   {len(df)} articles, {len(df.columns)} columns")
print(f"\nThis data can be used for text analysis in the data analysis block.")

---
## 🎯 Hands-on Tasks

**Task 1:** Run the notebook. Check the CSV — does it have titles, dates, summaries?

**Task 2:** Modify Cell 5 to extract an additional field (e.g., author name, category/tag, image URL).

**Task 3:** Try your own target. Ask AI to generate a scraping notebook for a website related to your research:

```
Create a Jupyter notebook that scrapes [what data] from [which website].
Use requests and BeautifulSoup. Handle errors. Export to CSV.
```

---

### Troubleshooting

| Problem | Solution |
|---------|----------|
| Status 403 | Site blocks bots. Try adding User-Agent header (Cell 2). |
| 0 articles found | HTML structure changed. Re-inspect with Developer Tools. |
| Fields show NO TITLE | Wrong selector. Check with Developer Tools. |
| Pagination doesn’t work | Check the URL pattern in your browser. |